# Fig.7-style capacity-change evaluation: QUIC wire throughput

**Main evaluation** (`--scenario fig7`, dynamic TBF on Path B server egress `h2-eth1`).

Compare **baseline** vs **qaccess_t_dynamic** from one session produced by `run_qaccess_t_fig7_dynamic_eval.sh`.

**Topology (static TCLink):**
- Path A / Link1: 20 Mbps, 40 ms, 0%
- Path B / Link2: 20 Mbps, 20 ms, 0.001%

**Dynamic capacity (Path B egress):** 0–50s @ 20 Mbps → 50–100s @ 30 Mbps → 100s+ @ 10 Mbps

**Primary figure:** total (Path A + Path B) server→client **QUIC wire throughput**.

Per-path and Path-B-vs-capacity plots are **diagnostics only** (not the main claim).

---

Tshark display filters use `udp`; all user-facing labels say **QUIC wire throughput**.


## 1. Setup

In [ ]:
import importlib.util
import subprocess
import sys

_PACKAGES = ["pandas", "matplotlib"]

def _have(mod: str) -> bool:
    return importlib.util.find_spec(mod) is not None

_missing = [p for p in _PACKAGES if not _have(p)]
if _missing:
    print("Installing:", ", ".join(_missing), flush=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-input", *_missing],
        timeout=600,
    )
else:
    print("OK:", ", ".join(_PACKAGES))


In [ ]:
import math
import os
import shutil
import subprocess
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("matplotlib", "inline")
except (ImportError, AttributeError):
    os.environ.setdefault("MPLBACKEND", "Agg")

try:
    from IPython.display import display
except ImportError:
    display = print


def find_repo() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "scripts" / "analyze" / "parse_logs.py").is_file():
            return p
    return cwd


REPO = find_repo()
print("REPO =", REPO)


## 2. Run paths

In [ ]:
# Set SESSION to logs_exp/session_fig7_dynamic_<timestamp> from run_qaccess_t_fig7_dynamic_eval.sh
SESSION = REPO / "logs_exp" / "session_fig7_dynamic_<timestamp>"

RUNS = {
    "baseline": SESSION / "fig7_baseline",
    "qaccess_t_dynamic": SESSION / "fig7_qaccess_t_dynamic",
}

OUT_DIR = SESSION / "compare_csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_A_DOWNLINK_FILTER = "udp && ip.src == 10.0.1.2 && ip.dst == 10.0.1.1"
PATH_B_DOWNLINK_FILTER = "udp && ip.src == 10.0.2.2 && ip.dst == 10.0.2.1"

SECOND_MAX = 220

WINDOWS = [
    ("0-50", 0, 50),
    ("50-60", 50, 60),
    ("50-100", 50, 100),
    ("100-110", 100, 110),
    ("100-150", 100, 150),
]

CSV_TIMESERIES = {
    "baseline": OUT_DIR / "throughput_quic_timeseries_baseline.csv",
    "qaccess_t_dynamic": OUT_DIR / "throughput_quic_timeseries_qaccess_t_dynamic.csv",
}
CSV_WINDOW_SUMMARY = OUT_DIR / "fig7_dynamic_quic_window_summary.csv"

FIG_TOTAL = OUT_DIR / "fig7_dynamic_total_quic_throughput.png"
FIG_WINDOW = OUT_DIR / "fig7_dynamic_window_total_quic_throughput.png"
FIG_PATHS_DIAG = OUT_DIR / "fig7_dynamic_per_path_quic_throughput_diagnostic.png"
FIG_PATHB_DIAG = OUT_DIR / "fig7_dynamic_pathB_downlink_diagnostic.png"

DEFAULT_INPUT_FLV = Path(os.path.expanduser("~/Videos/push_input.flv"))


def configured_path_b_capacity_mbps(second: int) -> float:
    if second < 50:
        return 20.0
    if second < 100:
        return 30.0
    return 10.0


def find_pcaps(run_dir: Path):
    pcaps = sorted((run_dir / "pcaps").glob("path*.pcap"))
    pcap_a = next(p for p in pcaps if "pathA" in p.name)
    pcap_b = next(p for p in pcaps if "pathB" in p.name)
    return pcap_a, pcap_b


def read_input_path_from_logs(run_dir: Path) -> str | None:
    logs = run_dir / "logs"
    if not logs.is_dir():
        return None
    for logf in sorted(logs.glob("*.log")):
        try:
            text = logf.read_text(errors="replace")
        except OSError:
            continue
        for line in text.splitlines():
            if "input   =" in line or "input =" in line:
                return line.split("=", 1)[1].strip()
    return None


print("Session path:", SESSION.resolve())
assert shutil.which("tshark"), "tshark not on PATH"

for label, run_dir in RUNS.items():
    pcap_a, pcap_b = find_pcaps(run_dir)
    print(f"\n[{label}] run:", run_dir)
    print("  Path A pcap:", pcap_a.name, "ok" if pcap_a.is_file() else "MISSING")
    print("  Path B pcap:", pcap_b.name, "ok" if pcap_b.is_file() else "MISSING")
    logged_input = read_input_path_from_logs(run_dir)
    print("  Input FLV (from logs):", logged_input or "(none — SAVE_LOGS=0 or not yet run)")


## 3. Pcap reader

In [ ]:
def read_frame_len_bins(pcap: Path, display_filter: str) -> defaultdict:
    bins = defaultdict(int)
    cmd = [
        "tshark", "-r", str(pcap), "-Y", display_filter,
        "-T", "fields", "-E", "separator=\t",
        "-e", "frame.time_relative", "-e", "frame.len",
    ]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or f"tshark failed: {pcap}")
    for line in proc.stdout.splitlines():
        parts = line.strip().split("\t")
        if len(parts) < 2:
            continue
        try:
            si = int(math.floor(float(parts[0])))
            flen = int(parts[1])
        except ValueError:
            continue
        if si >= 0:
            bins[si] += flen
    return bins


def bins_to_mbps_series(bin_dict: defaultdict, last_second: int, interval_s: float = 1.0) -> pd.Series:
    return pd.Series(
        [bin_dict.get(s, 0) * 8 / 1_000_000 / interval_s for s in range(0, last_second + 1)],
        dtype="float64",
    )


def mean_in_window(series: pd.Series, seconds: pd.Series, lo: int, hi: int) -> float:
    w = series[(seconds >= lo) & (seconds < hi)]
    return float(w.mean()) if len(w) else float("nan")


def improvement_pct(from_mbps: float, to_mbps: float) -> float:
    if from_mbps > 0 and math.isfinite(from_mbps) and math.isfinite(to_mbps):
        return (to_mbps - from_mbps) / from_mbps * 100.0
    return float("nan")


## 4. Build per-run timeseries

In [ ]:
def load_run_quic_timeseries(label: str, run_dir: Path) -> pd.DataFrame:
    pcap_a, pcap_b = find_pcaps(run_dir)
    print(f"Reading [{label}] pcaps for QUIC wire throughput...", flush=True)
    bins_a = read_frame_len_bins(pcap_a, PATH_A_DOWNLINK_FILTER)
    bins_b = read_frame_len_bins(pcap_b, PATH_B_DOWNLINK_FILTER)
    pcap_last = int(max(max(bins_a.keys(), default=0), max(bins_b.keys(), default=0)))
    last_s = int(min(SECOND_MAX, pcap_last))
    df = pd.DataFrame({
        "second": range(0, last_s + 1),
        "pathA_quic_mbps": bins_to_mbps_series(bins_a, last_s).values,
        "pathB_quic_mbps": bins_to_mbps_series(bins_b, last_s).values,
    })
    df["total_quic_mbps"] = df["pathA_quic_mbps"] + df["pathB_quic_mbps"]
    out_path = CSV_TIMESERIES[label]
    df.to_csv(out_path, index=False)
    print(f"  Wrote: {out_path} (rows={len(df)})")
    return df


run_dfs = {}
last_s = 0
for label, run_dir in RUNS.items():
    df = load_run_quic_timeseries(label, run_dir)
    run_dfs[label] = df
    last_s = max(last_s, int(df["second"].max()))

df_baseline = run_dfs["baseline"]
df_dynamic = run_dfs["qaccess_t_dynamic"]


## 5. Window summary (total QUIC wire throughput)

In [ ]:
window_defs = WINDOWS + [("full", 0, last_s + 1)]

summary_rows = []
for wname, lo, hi in window_defs:
    b_total = mean_in_window(df_baseline["total_quic_mbps"], df_baseline["second"], lo, hi)
    d_total = mean_in_window(df_dynamic["total_quic_mbps"], df_dynamic["second"], lo, hi)
    b_pa = mean_in_window(df_baseline["pathA_quic_mbps"], df_baseline["second"], lo, hi)
    d_pa = mean_in_window(df_dynamic["pathA_quic_mbps"], df_dynamic["second"], lo, hi)
    b_pb = mean_in_window(df_baseline["pathB_quic_mbps"], df_baseline["second"], lo, hi)
    d_pb = mean_in_window(df_dynamic["pathB_quic_mbps"], df_dynamic["second"], lo, hi)
    summary_rows.append({
        "window": wname,
        "baseline_total_mbps": round(b_total, 3),
        "qaccess_t_dynamic_total_mbps": round(d_total, 3),
        "improvement_pct": round(improvement_pct(b_total, d_total), 2),
        "baseline_pathA_mbps": round(b_pa, 3),
        "qaccess_t_dynamic_pathA_mbps": round(d_pa, 3),
        "baseline_pathB_mbps": round(b_pb, 3),
        "qaccess_t_dynamic_pathB_mbps": round(d_pb, 3),
    })

df_window_summary = pd.DataFrame(summary_rows)
df_window_summary.to_csv(CSV_WINDOW_SUMMARY, index=False)
print("Fig.7-style capacity-change — window summary (mean QUIC wire throughput, Mbps):")
print("Wrote", CSV_WINDOW_SUMMARY)
display(df_window_summary)


## 6. Main plots

In [ ]:
# 6a. PRIMARY: Total QUIC wire throughput over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_baseline["second"], df_baseline["total_quic_mbps"], label="baseline", linewidth=1.2)
ax.plot(df_dynamic["second"], df_dynamic["total_quic_mbps"], label="qaccess_t_dynamic", linewidth=1.2)
for t in (50, 100):
    ax.axvline(t, linestyle="--", linewidth=1, color="gray", alpha=0.7)
ax.set_xlabel("Time from pcap start (s)")
ax.set_ylabel("QUIC wire throughput (Mbps)")
ax.set_title("Fig.7-Style Capacity-Change Evaluation: Total QUIC Wire Throughput")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_TOTAL, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_TOTAL)


In [ ]:
# 6b. Window mean — total QUIC wire throughput only
plot_df = df_window_summary[df_window_summary["window"] != "full"].reset_index(drop=True)
x = range(len(plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width / 2 for i in x], plot_df["baseline_total_mbps"], width=width, label="baseline")
ax.bar([i + width / 2 for i in x], plot_df["qaccess_t_dynamic_total_mbps"], width=width, label="qaccess_t_dynamic")
ax.set_xticks(list(x))
ax.set_xticklabels(plot_df["window"].tolist(), rotation=25, ha="right")
ax.set_xlabel("Window")
ax.set_ylabel("Mean QUIC wire throughput (Mbps)")
ax.set_title("Fig.7-Style Evaluation: Window Mean Total QUIC Throughput")
ax.legend(loc="best")
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_WINDOW, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_WINDOW)


## 7. Diagnostic plots (per-path, optional)

In [ ]:
# 7a. Per-path downlink (diagnostic)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, col, title in [
    (axes[0], "pathA_quic_mbps", "Path A downlink (diagnostic)"),
    (axes[1], "pathB_quic_mbps", "Path B downlink (diagnostic)"),
]:
    ax.plot(df_baseline["second"], df_baseline[col], label="baseline", linewidth=1.0)
    ax.plot(df_dynamic["second"], df_dynamic[col], label="qaccess_t_dynamic", linewidth=1.0)
    for t in (50, 100):
        ax.axvline(t, linestyle="--", color="gray", alpha=0.6)
    ax.set_ylabel("QUIC wire throughput (Mbps)")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
axes[1].set_xlabel("Time from pcap start (s)")
fig.suptitle("Fig.7-Style Evaluation: Per-Path QUIC Wire Throughput (Diagnostic)", y=1.01, fontsize=11)
fig.tight_layout()
fig.savefig(FIG_PATHS_DIAG, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_PATHS_DIAG)


In [ ]:
# 7b. Path B downlink vs configured capacity (diagnostic — not the main claim)
cap = [configured_path_b_capacity_mbps(int(s)) for s in df_baseline["second"]]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_baseline["second"], df_baseline["pathB_quic_mbps"], label="baseline Path B", linewidth=1.0)
ax.plot(df_dynamic["second"], df_dynamic["pathB_quic_mbps"], label="qaccess_t_dynamic Path B", linewidth=1.0)
ax.plot(df_baseline["second"], cap, label="configured Path B capacity", linestyle="--", color="black")
for t in (50, 100):
    ax.axvline(t, linestyle=":", color="gray", alpha=0.6)
ax.set_xlabel("Time from pcap start (s)")
ax.set_ylabel("QUIC wire throughput (Mbps)")
ax.set_title("Diagnostic: Path B Downlink vs Configured Capacity")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_PATHB_DIAG, dpi=200, bbox_inches="tight")
plt.show()
print("Saved", FIG_PATHB_DIAG)


## 8. Summary

In [ ]:
logged_baseline = read_input_path_from_logs(RUNS["baseline"])
logged_dynamic = read_input_path_from_logs(RUNS["qaccess_t_dynamic"])

print("=" * 60)
print("Fig.7-style capacity-change evaluation — summary")
print("=" * 60)
print("Session:", SESSION.resolve())
print("Compares: baseline vs qaccess_t_dynamic only (no qaccess_t_static)")
print("Scenario: fig7 (Path A 20 Mbps static; Path B 20 Mbps static + TBF 20→30→10 on h2-eth1)")
print()
print("Input FLV path (baseline logs):", logged_baseline or "(not in logs)")
print("Input FLV path (dynamic logs):", logged_dynamic or "(not in logs)")
print("mp_topo default if --input-flv omitted:", DEFAULT_INPUT_FLV)
print()
print("Output CSV / figures under:", OUT_DIR.resolve())
for p in (FIG_TOTAL, FIG_WINDOW, CSV_WINDOW_SUMMARY):
    print(" ", p)
print()
print("Diagnostic only:", FIG_PATHS_DIAG, FIG_PATHB_DIAG)
print()
display(df_window_summary)
